# Week 56, Day 5: Mock interview

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w056d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from collections import Counter

text = "aaabbc"
counts = Counter(text)
most = max(counts.values())
print(counts.most_common(1), most, len(text) - most, most - 1 <= len(text) - most)

In [ ]:
import heapq
from collections import Counter

def rearrange(text):
    heap = [(-count, ch) for ch, count in Counter(text).items()]
    heapq.heapify(heap)
    out, held = [], None                     # held: the letter just used, waiting a turn
    while heap:
        count, ch = heapq.heappop(heap)
        out.append(ch)
        if held:
            heapq.heappush(heap, held)
        held = (count + 1, ch) if count + 1 < 0 else None
    return "".join(out) if len(out) == len(text) else None

print(rearrange("aaabbc"), rearrange("aaab"), rearrange(""))

In [ ]:
def longest_increasing_path(grid):
    R, C = len(grid), len(grid[0])
    cells = sorted(((grid[r][c], r, c) for r in range(R) for c in range(C)), reverse=True)
    best = [[1] * C for _ in range(R)]
    for value, r, c in cells:                    # largest first: their answers are ready
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] > value:
                best[r][c] = max(best[r][c], 1 + best[nr][nc])
    return max(max(row) for row in best)

print(longest_increasing_path([[9, 9, 4], [6, 6, 8], [2, 1, 1]]))

> **Key idea.** The seven steps stay the same; harder problems change what you say. Justify a greedy rule with an exchange argument, or reject it with a counterexample, and give a quick impossibility test if there is one. For DP, say the state, the recurrence, the order and the cost before coding. For graphs, say the states, the moves and what the state must remember, then pick BFS, Dijkstra or a topological order. For the space follow-up, use the input as workspace and mention that it changes the input.

### Exercise: Interview 1: no two alike

Write `rearrange(text)`: the letters of text in an order where no two neighbours are equal, or `None` if impossible, with the greedy heap (most frequent available letter first, ties alphabetical). Follow-up: `possible(text)`: True if such an order exists, in O(n) with one count, no heap.

In [ ]:
import heapq
from collections import Counter

def rearrange(text):
    ___

def possible(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\nfrom collections import Counter\ndef test_spread():\n    \"greedy with a heap\"\n    assert rearrange(\"aab\") == \"aba\" and rearrange(\"aaab\") is None and rearrange(\"\") == \"\" and rearrange(\"a\") == \"a\"\n    assert possible(\"aaabbc\") and not possible(\"aaab\") and possible(\"\")\n    rng = random.Random(1)\n    for _ in range(200):\n        s = \"\".join(rng.choice(\"aabbc\") for _ in range(rng.randint(0, 9)))\n        r = rearrange(s)\n        exists = any(all(x != y for x, y in zip(p, p[1:])) for p in set(itertools.permutations(s))) if len(s) <= 8 else None\n        if exists is not None:\n            assert (r is not None) == exists == possible(s)\n        if r is not None:\n            assert Counter(r) == Counter(s) and all(x != y for x, y in zip(r, r[1:]))\n    big = \"\".join(rng.choice(\"abcdef\") for _ in range(100_000))\n    r = rearrange(big)\n    assert r is not None and Counter(r) == Counter(big)")

### Exercise: Interview 2: the fewest terms

Courses 1 to n, with `before` pairs `(a, b)`: a must be finished in an **earlier** term than b. Any number of courses can be taken in one term. Write `fewest_terms(n, before)`: the fewest terms to finish everything, or `-1` if impossible. Follow-up: `term_plan(n, before)`: a list of terms, each a sorted list of courses, taking every course as early as possible.

In [ ]:
def fewest_terms(n, before):
    ___

def term_plan(n, before):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_terms():\n    \"Kahn's levels\"\n    assert fewest_terms(3, [(1, 3), (2, 3)]) == 2 and term_plan(3, [(1, 3), (2, 3)]) == [[1, 2], [3]]\n    assert fewest_terms(3, [(1, 2), (2, 3), (3, 1)]) == -1 and term_plan(3, [(1, 2), (2, 3), (3, 1)]) is None\n    assert fewest_terms(4, []) == 1 and fewest_terms(0, []) == 0\n    rng = random.Random(2)\n    for _ in range(100):\n        n = rng.randint(1, 8)\n        order = list(range(1, n + 1))\n        rng.shuffle(order)\n        pairs = list({(order[i], order[j]) for i in range(n) for j in range(i + 1, n) if rng.random() < 0.3})\n        plan = term_plan(n, pairs)\n        term_of = {c: t for t, courses in enumerate(plan) for c in courses}\n        assert sorted(term_of) == list(range(1, n + 1)) and all(term_of[a] < term_of[b] for a, b in pairs)\n        depth = {}\n        for c in order:\n            depth[c] = 1 + max([depth[a] for a, b in pairs if b == c] + [0])\n        assert fewest_terms(n, pairs) == max(depth.values()) == len(plan)")

### Exercise: Interview 3: always uphill

Write `longest_uphill(grid)`: the length of the longest path through a grid of numbers moving up, down, left or right, always to a **strictly larger** number. Follow-up: `count_uphill(grid)`: how many such paths there are (of any length, including single cells), mod 10⁹ + 7. Both by DP over cells sorted by value, with no recursion.

In [ ]:
def longest_uphill(grid):
    ___

def count_uphill(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(grid):\n    R, C = len(grid), len(grid[0])\n    stats = {\"longest\": 0, \"paths\": 0}\n    def go(r, c, length):\n        stats[\"paths\"] += 1\n        stats[\"longest\"] = max(stats[\"longest\"], length)\n        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):\n            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] > grid[r][c]:\n                go(nr, nc, length + 1)\n    for r in range(R):\n        for c in range(C):\n            go(r, c, 1)\n    return stats[\"longest\"], stats[\"paths\"]\ndef test_uphill():\n    \"DP on a DAG of cells\"\n    g = [[9, 9, 4], [6, 6, 8], [2, 1, 1]]\n    assert longest_uphill(g) == 4 and longest_uphill([[1]]) == 1 and count_uphill([[1, 2]]) == 3\n    rng = random.Random(3)\n    for _ in range(80):\n        R, C = rng.randint(1, 4), rng.randint(1, 4)\n        grid = [[rng.randint(0, 9) for _ in range(C)] for _ in range(R)]\n        L, N = brute(grid)\n        assert longest_uphill(grid) == L and count_uphill(grid) == N % (10 ** 9 + 7)\n    big = [[r * 300 + c for c in range(300)] for r in range(300)]\n    assert longest_uphill(big) == 599")

### Exercise: Interview 4: the first missing positive

Write `first_missing_positive(nums)`: the smallest positive whole number not in the list, in O(n) time and **O(1) extra space**, by swapping each value v from 1 to n into position v − 1 (the list may be changed). No sets, dicts, sorting or extra lists.

In [ ]:
def first_missing_positive(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_missing():\n    \"the input as workspace\"\n    assert first_missing_positive([1, 2, 0]) == 3 and first_missing_positive([3, 4, -1, 1]) == 2\n    assert first_missing_positive([7, 8, 9]) == 1 and first_missing_positive([]) == 1 and first_missing_positive([1, 1]) == 2\n    rng = random.Random(4)\n    for _ in range(300):\n        nums = [rng.randint(-3, 10) for _ in range(rng.randint(0, 10))]\n        want = next(k for k in range(1, 20) if k not in nums)\n        assert first_missing_positive(list(nums)) == want\n    big = list(range(1, 200_001))\n    rng.shuffle(big)\n    gone, big[123] = big[123], -5\n    assert first_missing_positive(big) == gone\n    assert \"set(\" not in source and \"sorted(\" not in source and \".sort(\" not in source and \"{\" not in source")

### Bug hunt: Testing your own code: the held letter

A used letter must wait **one turn** before going back into the heap. This version pushes it back straight away (when anything else is left), so the same letter can be chosen twice in a row: for `"aab"` it prints `'aab'`, two a's together. Hold the letter for a turn, pushing back the previous one instead, so it prints `['aba', 'bab']`.

In [ ]:
import heapq
from collections import Counter

def rearrange(text):
    heap = [(-n, ch) for ch, n in Counter(text).items()]
    heapq.heapify(heap)
    out, held = [], None
    while heap:
        n, ch = heapq.heappop(heap)
        out.append(ch)
        held = (n + 1, ch) if n + 1 < 0 else None
        if held and heap:
            heapq.heappush(heap, held)
    return "".join(out) if len(out) == len(text) else None

print([rearrange("aab"), rearrange("abb")])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"wait one turn\"\n    expect_output(\"['aba', 'bab']\")")

### Bug hunt: Testing your own code: swapping in a loop

The first-missing-positive swap must repeat **while** the value at i still belongs elsewhere: one swap may bring in another value that needs moving. With `if` instead of `while`, this prints `1` for `[3, 4, -1, 1]`; it should print `2`.

In [ ]:
def first_missing_positive(nums):
    n = len(nums)
    for i in range(n):
        if 1 <= nums[i] <= n and nums[nums[i] - 1] != nums[i]:
            j = nums[i] - 1
            nums[i], nums[j] = nums[j], nums[i]
    for i in range(n):
        if nums[i] != i + 1:
            return i + 1
    return n + 1

print(first_missing_positive([3, 4, -1, 1]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keep swapping\"\n    expect_output(\"2\")")

### Exercise: Predict them



In [ ]:
p = ___   # possible("aaabb"): True or False
q = ___   # fewest_terms(4, [(1, 2), (2, 3), (1, 4)])
r = ___   # first_missing_positive([2, 3, 4])
s = ___   # the cost of longest_uphill on an R × C grid, ignoring the sort: "O(R + C)", "O(R × C)" or "O((R × C)²)"
t = ___   # True or False: an exchange argument shows a greedy rule is wrong

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q == 3 and r == 1, \"p, q and r\"\n    assert s == \"O(R \u00d7 C)\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a rate limiter

A design question often closes an interview: "Build a rate limiter: each user may make at most `limit` requests in any `window` seconds." Talk through the options (a counter per fixed minute is simple but allows bursts at the boundary; a log of recent times per user is exact), then write the exact one.

- `RateLimiter(limit, window)`; `allow(user, t)`: record a request at time t (times for one user never decrease) and return True if it's allowed, i.e. fewer than `limit` of that user's **allowed** requests fall in the half-open window (t − window, t]. Refused requests aren't recorded.
- Each user keeps a `deque` of allowed times, dropping those that are too old, so each call is O(1) amortised.
- `forget_idle(t)`: delete every user whose most recent allowed request is at or before t − window (they have nothing left to remember), returning how many were removed. Memory then stays proportional to active users.
- `count(user, t)`: how many of the user's allowed requests are in the window ending at t.

In [ ]:
from collections import deque

class RateLimiter:
    def __init__(self, limit, window):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_limiter():\n    \"a sliding log\"\n    r = RateLimiter(3, 10)\n    assert [r.allow(\"ana\", t) for t in (1, 2, 3, 4, 11, 12, 13, 14)] == [True, True, True, False, True, True, True, False]\n    assert r.count(\"ana\", 14) == 3 and r.count(\"ana\", 30) == 0 and r.count(\"bo\", 5) == 0\n    assert r.allow(\"bo\", 14) and r.forget_idle(25) == 2 and r.forget_idle(25) == 0\n    r2 = RateLimiter(1, 5)\n    assert r2.allow(\"x\", 0) and not r2.allow(\"x\", 4) and r2.allow(\"x\", 5)\n    rng = random.Random(6)\n    for _ in range(30):\n        limit, window = rng.randint(1, 4), rng.randint(1, 10)\n        lim = RateLimiter(limit, window)\n        allowed = {u: [] for u in \"abc\"}\n        times = {u: 0 for u in \"abc\"}\n        for _ in range(60):\n            u = rng.choice(\"abc\")\n            times[u] += rng.randint(0, 4)\n            t = times[u]\n            recent = [x for x in allowed[u] if t - window < x <= t]\n            ok = len(recent) < limit\n            assert lim.allow(u, t) == ok\n            if ok:\n                allowed[u].append(t)\n    big = RateLimiter(5, 100)\n    for i in range(100_000):\n        big.allow(f\"u{i % 1000}\", i)\n    assert big.forget_idle(100_100) == 1000")

## The method, briefly

From [Week 49's mock interview](w049d5.html): **understand** (ask about sizes and awkward cases), **examples** (one ordinary, one awkward), **brute force** and its cost, **improve** by naming the repeated work and the pattern that removes it, **code**, **test** out loud with your examples, **analyse** time and space. Today's harder problems don't change the steps; they change what you have to say at step 4.

## Explaining a greedy choice

When you propose a greedy rule, the interviewer's next question is almost always "why does that work?" Have one of two answers ready:

- **An exchange argument** (Week 52): "Suppose an optimal answer doesn't start with my choice. Swap its first step for mine; the result is no worse. So some optimal answer starts my way."
- **A counterexample**, if you're testing a rule: "Taking the biggest coin first fails for coins 1, 3, 4 and amount 6, so this needs DP."

Example: rearrange a string so no two equal letters are next to each other. "Greedily, at each step place the most frequent remaining letter that isn't the one just placed. Why? Leaving the most frequent letter for later can only make it harder to space out. And it's possible exactly when the most frequent letter appears at most ⌈n / 2⌉ times." That last sentence, a quick test for impossibility, is the kind of insight that makes an interview go well.

## Saying a recurrence out loud

For a DP problem, say the state and recurrence **before** writing code, in the Week 53 form: "Let f(i, j) be the longest increasing path **starting** at cell (i, j). From (i, j) I can step to a larger neighbour, so f(i, j) = 1 + the largest f over larger neighbours, or 1 if there are none. The answer is the largest f anywhere."

Then answer the two questions that follow:

- **"Is there an order?"** Here, yes: moving only to larger values, there are no cycles, so memoisation works (or sorting cells by value gives a fill order).
- **"What does it cost?"** States × work: R × C cells, four neighbours each: O(R × C).

Interviewers often stop you before the code if the recurrence is right; the code is the easy part once you've said it.

## Graphs in disguise

"Rooms and keys", "courses and prerequisites", "a grid where you can break one wall", "the fewest semesters to finish": each is a graph. Say it: "The states are ..., the moves are ..., the moves all cost the same, so BFS." Then say what the state must contain: the cell **and** the walls broken, the room **and** the keys held. Getting the state right is half of a graph interview, as it is for DP.

The fewest semesters to finish every course, taking any number of courses whose prerequisites are done: that's the number of **levels** in Kahn's algorithm (Week 50): each round takes every course with no unfinished prerequisites.

## The space follow-up

"Can you do it with O(1) extra space?" usually means: **use the input as your workspace**. The classic is the first missing positive number. With a set it's O(n) space. Without one: the answer is between 1 and n + 1, so put each value v from 1 to n into position v − 1 by swapping; then the first position i not holding i + 1 gives the answer. Each swap puts one value in its final place, so it's O(n) time.

Say the trade-off aloud: "this changes the input; if the caller needs it unchanged, I'd copy it, which brings back O(n) space."

## Practice

Before coding each problem, write a comment block: your assumptions, two examples, the brute force and its cost, and your improved idea and its cost.

## Quiz

## Daily challenge

## Before you go

Pick one problem from this week's timed sets and talk it through out loud from start to finish, as if to an interviewer, timing yourself. Tomorrow: the debrief of timed sets 4 and 5.